In [ ]:
%stop_session

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore",
  "--enable-glue-datacatalog": "true"
}

### Setup

In [ ]:
#Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import DeltaTable
from datetime import *
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException

BASE = "s3://data-lake-case-hotmart"

# D-1 
RUN_DATE = (date.today() - timedelta(days=1)).isoformat() #D-1 troque para a data que deseja navegar
# uat : RUN_DATE = "2023-03-31"   # teste também: "2023-01-23", "2023-02-28", "2023-07-15"

PATH_BRONZE_PURCHASE         = f"{BASE}/bronze/purchase"
PATH_BRONZE_PRODUCT_ITEM     = f"{BASE}/bronze/product_item"
PATH_BRONZE_EXTRA_INFO       = f"{BASE}/bronze/purchase_extra_info"

PATH_SILVER_PURCHASE         = f"{BASE}/silver/purchase_daily"
PATH_SILVER_PRODUCT_ITEM     = f"{BASE}/silver/product_item_daily"
PATH_SILVER_EXTRA_INFO       = f"{BASE}/silver/purchase_extra_info_daily"

PATH_GOLD                    = f"{BASE}/gold/gmv_daily"
PATH_GOLD_GMV                = f"{BASE}/gold/gmv_daily_by_subsidiary"

In [ ]:
# Ler Silvers (DELTA)
df_purchase_silver = spark.read.format("delta").load(PATH_SILVER_PURCHASE)
df_product_item_silver = spark.read.format("delta").load(PATH_SILVER_PRODUCT_ITEM)
df_purchase_extra_info_silver = spark.read.format("delta").load(PATH_SILVER_EXTRA_INFO)

# snapshot_date = D-1
df_snapshot_date = spark.sql("SELECT date_sub(current_date(), 1) AS snapshot_date").collect()[0]["snapshot_date"]
print("snapshot_date (D-1):", df_snapshot_date)

In [ ]:
# -------------------------------------------------------------
# GOLD GVM (nível de compra) - foto "as-of" snapshot_date
# Para garantir reprodutibilidade e imutabilidade:
#   sempre escolhemos o último estado com transaction_date <= snapshot_date.
# -------------------------------------------------------------

# Purchase as-of
w_purchase = Window.partitionBy("purchase_id").orderBy(
    col("transaction_date").desc(),
    col("transaction_datetime").desc()
)
df_purchase_asof = (
    df_purchase_silver
    .filter(col("transaction_date") <= lit(df_snapshot_date))
    .withColumn("rn", row_number().over(w_purchase))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Product item as-of (por purchase_id + product_id)
w_item = Window.partitionBy("purchase_id","product_id").orderBy(
    col("transaction_date").desc(),
    col("transaction_datetime").desc()
)
df_item_asof = (
    df_product_item_silver
    .filter(col("transaction_date") <= lit(df_snapshot_date))
    .withColumn("rn", row_number().over(w_item))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Extra info as-of (por purchase_id)
w_extra = Window.partitionBy("purchase_id").orderBy(
    col("transaction_date").desc(),
    col("transaction_datetime").desc()
)
df_extra_asof = (
    df_purchase_extra_info_silver
    .filter(col("transaction_date") <= lit(df_snapshot_date))
    .withColumn("rn", row_number().over(w_extra))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Compras pagas (GMV considera somente release_date preenchida)
df_purchase_asof = df_purchase_asof.filter(col("transaction_status") == "Succesfull")

# Join assíncrono:
# - se item ainda não chegou, purchase_value/item_quantity ficam null e a compra não entra no GMV
# - se extra ainda não chegou, subsidiary fica null (vamos tratar como UNKNOWN na agregação)
df_new_gvm = (
    df_purchase_asof.alias("a")
    .join(df_item_asof.alias("b"), col("a.purchase_id") == col("b.purchase_id"), "left")
    .join(df_extra_asof.alias("c"), col("a.purchase_id") == col("c.purchase_id"), "left")
    .select(
        col("a.transaction_datetime"),
        col("a.purchase_id"),
        col("a.buyer_id"),
        col("a.prod_item_id"),
        col("a.order_date"),
        col("a.release_date"),
        col("a.producer_id"),
        col("b.product_id"),
        col("b.item_quantity"),
        col("b.purchase_value"),
        col("c.subsidiary"),
        current_timestamp().alias("snapshot_datetime"),
        lit(df_snapshot_date).cast("date").alias("snapshot_date"),
        col("a.transaction_date")
    )
)

df_new_gvm.show(truncate=False)

In [ ]:
# -------------------------------------------------------------
# Persistência do GOLD GVM (DELTA) - idempotente por snapshot_date
# Estratégia simples e compatível com o notebook original:
#  - lê o gold existente (se houver)
#  - union + dedup do snapshot do dia
#  - recalcula is_current_snapshot com base no max(snapshot_date)
#  - overwrite (mantendo histórico)
# -------------------------------------------------------------
try:
    df_gold_existing = spark.read.format("delta").load(PATH_GOLD)
    # compatibilidade com versões antigas: remove current_snapshot se existir
    if "current_snapshot" in df_gold_existing.columns:
        df_gold_existing = df_gold_existing.drop("current_snapshot")
    df_gold_union = df_gold_existing.unionByName(df_new_gvm, allowMissingColumns=True)
except AnalysisException:
    df_gold_union = df_new_gvm

w_snap = Window.partitionBy("purchase_id","snapshot_date").orderBy(col("snapshot_datetime").desc())
df_gold_dedup = (
    df_gold_union
    .withColumn("rn", row_number().over(w_snap))
    .filter(col("rn") == 1)
    .drop("rn")
)

max_snapshot = df_gold_dedup.select(max("snapshot_date").alias("mx")).collect()[0]["mx"]
df_gold_final = df_gold_dedup.withColumn("is_current_snapshot", col("snapshot_date") == lit(max_snapshot))

df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("snapshot_date","transaction_date") \
    .save(PATH_GOLD)

df_gold_final.createOrReplaceTempView("gvm_gold")
spark.sql("SELECT * FROM gvm_gold WHERE is_current_snapshot = true ORDER BY purchase_id").show(truncate=False)

In [ ]:
# -------------------------------------------------------------
# DATASET FINAL (entregável): GMV diário por subsidiária
# - regra GMV: sum(item_quantity * purchase_value)
# - particiona por transaction_date (usamos snapshot_date para garantir D-1)
# - is_current_snapshot facilita consumo (usuário não precisa de subquery de MAX)
# -------------------------------------------------------------
df_gmv_daily = spark.sql("""
    SELECT
        snapshot_date,
        release_date,
        COALESCE(subsidiary, 'UNKNOWN') AS subsidiary,
        SUM(item_quantity * purchase_value) AS gmv_value,
        COUNT(DISTINCT purchase_id) AS transaction_count,
        is_current_snapshot,
        snapshot_date AS transaction_date
    FROM gvm_gold
    GROUP BY 1,2,3,6,7
""")

df_gmv_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .save(PATH_GOLD_GMV)

df_gmv_daily.show(truncate=False)